In [ ]:
!pip install transformers seqeval evaluate accelerate -U
!pip install transformers seqeval evaluate accelerate pytorch-crf -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=b4391edd855c113a901d06b576d2a31b4f5b4e3e54de20d2420a5cf0cd9842bf
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import numpy as np
import random
from transformers import set_seed

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Đảm bảo cudnn deterministic nếu cần thiết
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'
train_path = '/content/drive/MyDrive/datasetViMedNER/traindata/train.txt'
dev_path = '/content/drive/MyDrive/datasetViMedNER/traindata/dev.txt'

unique_tags = []
with open(label_path, "r", encoding = "utf-8") as f :
    for line in f:
        line.strip()
        if line.strip():
          unique_tags.append(line.strip())

label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

def load_conll_data(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
            else:
                parts = line.split()
                if len(parts) >= 2:
                    word, tag = parts[0], parts[1]
                    current_sentence.append((word, tag))
        if current_sentence:
            sentences.append(current_sentence)
    return sentences

train_sentences = load_conll_data(train_path)
dev_sentences = load_conll_data(dev_path)

In [ ]:
import numpy as np
import pandas as pd
import evaluate
from sklearn.metrics import classification_report as sklearn_report
from IPython.display import display

seqeval = evaluate.load("seqeval")

def compute_eval_classify_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    # 1. Lọc nhãn -100 và giữ nguyên cấu trúc List of Lists (dành cho Seqeval)
    true_predictions_seq = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels_seq = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 2. Trải phẳng thành mảng 1 chiều (dành cho Sklearn)
    true_predictions_flat = [tag for sent in true_predictions_seq for tag in sent]
    true_labels_flat = [tag for sent in true_labels_seq for tag in sent]

    metrics_dict = {}

    # =========================================================
    # BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ HOÀN CHỈNH (SEQEVAL)
    # =========================================================
    results_seq = seqeval.compute(predictions=true_predictions_seq, references=true_labels_seq)
    table_entity = []

    for key, value in results_seq.items():
        if isinstance(value, dict):
            table_entity.append({
                "Thực thể (Entity)": key,
                "Precision": f"{value['precision']:.4f}",
                "Recall": f"{value['recall']:.4f}",
                "F1-Score": f"{value['f1']:.4f}",
                "Number (Entities)": value['number']
            })
            metrics_dict[f"entity_{key}_f1"] = value["f1"]

    table_entity.append({
        "Thực thể (Entity)": "OVERALL",
        "Precision": f"{results_seq['overall_precision']:.4f}",
        "Recall": f"{results_seq['overall_recall']:.4f}",
        "F1-Score": f"{results_seq['overall_f1']:.4f}",
        "Number (Entities)": "-"
    })
    metrics_dict["overall_f1"] = results_seq['overall_f1']

    print("\n" + "="*75)
    print("📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)")
    print("="*75)
    display(pd.DataFrame(table_entity))

    # =========================================================
    # BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN RỜI RẠC (SKLEARN)
    # =========================================================
    report_tag = sklearn_report(true_labels_flat, true_predictions_flat, output_dict=True, zero_division=0)
    table_tag = []

    for key, value in report_tag.items():
        if key in ["accuracy", "macro avg", "weighted avg"]:
            continue
        table_tag.append({
            "Nhãn (Tag)": key,
            "Precision": f"{value['precision']:.4f}",
            "Recall": f"{value['recall']:.4f}",
            "F1-Score": f"{value['f1-score']:.4f}",
            "Number (Tokens)": int(value['support'])
        })

    overall_tag = report_tag["weighted avg"]
    table_tag.append({
        "Nhãn (Tag)": "OVERALL (Weighted)",
        "Precision": f"{overall_tag['precision']:.4f}",
        "Recall": f"{overall_tag['recall']:.4f}",
        "F1-Score": f"{overall_tag['f1-score']:.4f}",
        "Number (Tokens)": int(overall_tag['support'])
    })

    print("\n" + "="*75)
    print("📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)")
    print("="*75)
    display(pd.DataFrame(table_tag))

    return metrics_dict

In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    # Rút trích riêng điểm của nguyen_nhan_benh
    f1_nguyen_nhan = 0.0
    if "nguyen_nhan_benh" in results:
        f1_nguyen_nhan = results["nguyen_nhan_benh"]["f1"]

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
        "f1_NNB": f1_nguyen_nhan  # Thêm cột F1 NNB vào log
    }

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
import pickle
train_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/train_dataset.pkl'
dev_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'
test_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/test_dataset.pkl'

def load_dataset(file_path):
    with open(file_path,'rb') as f :
        dataset = pickle.load(f)
    return dataset


train_dataset = load_dataset(train_dataset_filepath)
dev_dataset = load_dataset(dev_dataset_filepath)
test_dataset = load_dataset(test_dataset_filepath)

In [ ]:
sentence_lengths = []
entity_lengths = []
all_tags = set()
tag_counts = {}

for sent in train_sentences:
    sentence_lengths.append(len(sent))

    # Trích xuất thực thể từ chuỗi B- / I-
    in_entity = False
    current_ent_len = 0

    for word, tag in sent:
        all_tags.add(tag)
        tag_counts[tag] = tag_counts.get(tag, 0) + 1

        if tag.startswith('B-'):
            if in_entity:
                entity_lengths.append(current_ent_len)
            in_entity = True
            current_ent_len = 1
        elif tag.startswith('I-') and in_entity:
            current_ent_len += 1
        else:
            if in_entity:
                entity_lengths.append(current_ent_len)
                in_entity = False
                current_ent_len = 0
    if in_entity:
        entity_lengths.append(current_ent_len)

In [ ]:
label2id

{'B-bien_phap_chan_doan': 0,
 'B-bien_phap_dieu_tri': 1,
 'B-nguyen_nhan_benh': 2,
 'B-ten_benh': 3,
 'B-trieu_chung_benh': 4,
 'I-bien_phap_chan_doan': 5,
 'I-bien_phap_dieu_tri': 6,
 'I-nguyen_nhan_benh': 7,
 'I-ten_benh': 8,
 'I-trieu_chung_benh': 9,
 'O': 10}

In [ ]:
import math
import torch
import numpy as np

# Giả định Thành đã chạy sẵn tag_counts và label2id ở trên
sum_tag = sum(tag_counts.values())

# 1. Tính toán w_freq (Tần suất) và w_diff (Độ khó)
w_freq = np.zeros(len(label2id))
for tag, idx in label2id.items():
    w_freq[idx] = math.sqrt(sum_tag / tag_counts[tag])

f1_scores = {
    0: 0.7592, 1: 0.7722, 2: 0.1843, 3: 0.8845, 4: 0.7457,
    5: 0.7639, 6: 0.7350, 7: 0.4994, 8: 0.9168, 9: 0.7442,
    10: 0.9569
}

# Lấy điểm F1 cao nhất của nhóm thực thể (loại nhãn O)
f1_max = max(list(f1_scores.values())[:-1])

w_diff = np.zeros(len(label2id))
for tag, idx in label2id.items():
    w_diff[idx] = math.sqrt(f1_max / f1_scores[idx])

# 2. Tính static weight thô
static_weight_raw = w_freq * w_diff

# 3. Tinh chỉnh Scale (Chỉ xét min/max trên nhóm thực thể)
O_index = label2id['O']
entity_weights = np.delete(static_weight_raw, O_index)

min_w = np.min(entity_weights)
max_w = np.max(entity_weights)

target_min = 0.5
target_max = 3.0

static_weight_scaled = np.zeros_like(static_weight_raw)

for idx in range(len(static_weight_raw)):
    if idx == O_index:
        continue
    # Công thức Min-Max chuẩn: giữ nguyên tỷ lệ thuận (khó -> to, dễ -> nhỏ)
    static_weight_scaled[idx] = target_min + (static_weight_raw[idx] - min_w) * (target_max - target_min) / (max_w - min_w)

# 4. Ép cứng nhãn O xuống đáy (0.1)
static_weight_scaled[O_index] = 0.1

print("\n--- BẢNG TRỌNG SỐ SAU KHI SCALE CHUẨN ---")
for tag, idx in label2id.items():
    print(f"{idx:2d} | {tag:25s} | Weight = {static_weight_scaled[idx]:.4f}")

# 5. Chuyển thành PyTorch Tensor sẵn sàng dùng cho model
static_weight_tensor = torch.tensor(static_weight_scaled, dtype=torch.float)


--- BẢNG TRỌNG SỐ SAU KHI SCALE CHUẨN ---
 0 | B-bien_phap_chan_doan     | Weight = 1.5579
 1 | B-bien_phap_dieu_tri      | Weight = 1.1587
 2 | B-nguyen_nhan_benh        | Weight = 3.0000
 3 | B-ten_benh                | Weight = 0.6931
 4 | B-trieu_chung_benh        | Weight = 1.0335
 5 | I-bien_phap_chan_doan     | Weight = 0.9864
 6 | I-bien_phap_dieu_tri      | Weight = 0.7826
 7 | I-nguyen_nhan_benh        | Weight = 1.0261
 8 | I-ten_benh                | Weight = 0.5000
 9 | I-trieu_chung_benh        | Weight = 0.7751
10 | O                         | Weight = 0.1000


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF

class EntityAware_ViMedNER(nn.Module):
    def __init__(self, model_checkpoint, num_labels, static_weights, lambda_weight= 0.75, gamma_len=1.0):
        super().__init__()
        self.num_labels = num_labels
        self.lambda_weight = lambda_weight
        self.gamma_len = gamma_len
        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.register_buffer('w_class', torch.tensor(static_weights, dtype=torch.float))

        # Thêm biến đếm để in log Loss
        self.batch_counter = 0

    def _get_span_weight_mask(self, labels):
        batch_size, seq_len = labels.shape
        mask = torch.ones((batch_size, seq_len), device=labels.device)

        O_LABEL_ID = 10
        # Khai báo danh sách các nhãn B- (dựa trên label2id)
        B_LABEL_IDS = [0, 1, 2, 3, 4]

        for i in range(batch_size):
            seq_labels = labels[i].tolist()
            start = -1
            for j in range(seq_len):
                lbl = seq_labels[j]

                # LOGIC MỚI: Cắt cụm nếu gặp O, PAD hoặc bắt đầu một nhãn B- mới
                if lbl == -100 or lbl == O_LABEL_ID or lbl in B_LABEL_IDS:
                    if start != -1:
                        # Kết thúc cụm cũ, tính w_len và gán
                        span_len = j - start
                        w_len = 1.0 + self.gamma_len * math.log(1.0 + span_len)
                        mask[i, start:j] = w_len
                        start = -1

                # Khởi tạo cụm mới nếu token là thực thể (B- hoặc I-)
                if lbl != -100 and lbl != O_LABEL_ID:
                    if start == -1:
                        start = j

            # Chốt sổ đoạn cuối câu
            if start != -1:
                span_len = seq_len - start
                w_len = 1.0 + self.gamma_len * math.log(1.0 + span_len)
                mask[i, start:seq_len] = w_len
        return mask

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        emissions = self.classifier(sequence_output)

        crf_mask = attention_mask.bool()
        crf_mask[:, 0] = True
        loss = None

        if labels is not None:
            # 1. TÍNH MAIN LOSS: CRF
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 10
            l_crf = -self.crf(emissions, safe_labels, mask=crf_mask, reduction='mean')

            # 2. TÍNH AUXILIARY LOSS: Entity-Aware
            l_aux_raw = F.cross_entropy(
                emissions.view(-1, self.num_labels),
                labels.view(-1),
                weight=self.w_class,
                ignore_index=-100,
                reduction='none'
            )

            w_len_mask = self._get_span_weight_mask(labels).view(-1)
            valid_tokens_mask = (labels != -100).view(-1)
            l_aux_weighted = (l_aux_raw * w_len_mask).sum() / (valid_tokens_mask.sum() + 1e-9)

            # 3. TỔNG LOSS
            loss = l_crf + (self.lambda_weight * l_aux_weighted)

        crf_mask_decode = attention_mask.bool()
        decoded_paths = self.crf.decode(emissions, mask=crf_mask_decode)
        fake_logits = torch.zeros_like(emissions)
        for i, path in enumerate(decoded_paths):
            for j, tag_id in enumerate(path):
                fake_logits[i, j, tag_id] = 1.0

        return TokenClassifierOutput(loss=loss, logits=fake_logits)

In [ ]:
from torch.optim import AdamW
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
# 1. Khởi tạo mô hình Entity-Aware CRF mới của Thành
model_entity_aware = EntityAware_ViMedNER(
    model_checkpoint="vinai/phobert-base-v2",
    num_labels=len(label2id),
    static_weights=static_weight_tensor, # Sử dụng tensor trọng số đã tính ở bước trước
    lambda_weight=0.75,
    gamma_len=1.0
)

# 2. Tách nhóm tham số và gán Learning Rate riêng biệt (BÍ KÍP TỐI ƯU)
crf_params = list(model_entity_aware.crf.parameters())
# Gom toàn bộ phần PhoBERT, Linear classifier và các trọng số khác vào nhóm 1
base_classifier_params = [p for n, p in model_entity_aware.named_parameters() if not n.startswith("crf.")]

optimizer_grouped_parameters = [
    {'params': base_classifier_params, 'lr': 3e-5}, # LR nhỏ bảo vệ PhoBERT
    {'params': crf_params, 'lr': 3e-3}             # LR lớn tăng tốc học cho CRF
]
optimizer_entity_aware = AdamW(optimizer_grouped_parameters, weight_decay=0.01)

# 3. Cấu hình Training Arguments chuẩn chỉnh
training_args_aware = TrainingArguments(
    output_dir="./entity_aware_vimedner_model",
    eval_strategy="epoch",
    learning_rate=3e-5, # Giá trị mặc định chung, sẽ bị ghi đè bởi custom optimizer bên trên
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    max_grad_norm=1.0,
    report_to="none"
)

# 4. Khởi tạo Trainer hoàn chỉnh kèm Early Stopping và Custom Optimizer
trainer_entity_aware = Trainer(
    model=model_entity_aware,
    args=training_args_aware,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer_entity_aware, None), # Cắm optimizer tách biệt LR vào đây
    callbacks=[EarlyStoppingCallback(early_stopping_patience=6)]
)

# 5. Bắt đầu huấn luyện
trainer_entity_aware.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_2408/4126588874.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer('w_class', torch.tensor(static_weights, dtype=torch.float))


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,F1 Nnb
1,No log,12.247107,0.652762,0.634362,0.643431,0.883061,0.015152
2,17.590105,8.919042,0.661261,0.689664,0.675164,0.890592,0.103152
3,17.590105,8.244246,0.691781,0.704966,0.698311,0.898141,0.182353
4,6.684483,7.889945,0.648780,0.714094,0.679872,0.877648,0.189944
5,6.684483,7.529911,0.676938,0.731275,0.703058,0.894438,0.270169
6,4.126649,8.202756,0.680803,0.728322,0.703761,0.892996,0.335155
7,2.797023,8.965229,0.719979,0.722685,0.721329,0.901595,0.325482
8,2.797023,9.602808,0.658462,0.746846,0.699874,0.889453,0.281046
9,1.916142,9.623514,0.691688,0.737181,0.713710,0.896717,0.315992
10,1.916142,10.580722,0.689950,0.737181,0.712784,0.896147,0.331429


TrainOutput(global_step=3718, training_loss=4.827844152917395, metrics={'train_runtime': 2657.4084, 'train_samples_per_second': 25.813, 'train_steps_per_second': 1.614, 'total_flos': 0.0, 'train_loss': 4.827844152917395, 'epoch': 13.0})

In [ ]:
pred_results = trainer_entity_aware.predict(dev_dataset)

compute_eval_classify_metrics((pred_results.predictions, pred_results.label_ids))
compute_metrics((pred_results.predictions, pred_results.label_ids))


📊 BẢNG 1: ĐÁNH GIÁ THEO CỤM THỰC THỂ (STRICT ENTITY LEVEL)


,Thực thể (Entity),Precision,Recall,F1-Score,Number (Entities)
0,bien_phap_chan_doan,0.6958,0.6753,0.6854,271
1,bien_phap_dieu_tri,0.6204,0.6233,0.6218,653
2,nguyen_nhan_benh,0.3654,0.2934,0.3255,259
3,ten_benh,0.8101,0.8630,0.8357,1795
4,trieu_chung_benh,0.6814,0.6386,0.6593,747
5,OVERALL,0.7200,0.7227,0.7213,-



📊 BẢNG 2: ĐÁNH GIÁ CHI TIẾT TỪNG NHÃN (TOKEN TAG LEVEL: B-, I-, O)


,Nhãn (Tag),Precision,Recall,F1-Score,Number (Tokens)
0,B-bien_phap_chan_doan,0.7558,0.7196,0.7372,271
1,B-bien_phap_dieu_tri,0.7034,0.6937,0.6985,653
2,B-nguyen_nhan_benh,0.4660,0.3707,0.4129,259
3,B-ten_benh,0.8449,0.8986,0.8710,1795
4,B-trieu_chung_benh,0.7378,0.6894,0.7128,747
5,I-bien_phap_chan_doan,0.7739,0.5594,0.6494,942
6,I-bien_phap_dieu_tri,0.6653,0.5524,0.6036,1745
7,I-nguyen_nhan_benh,0.4470,0.3424,0.3877,850
8,I-ten_benh,0.8792,0.9156,0.8970,4585
9,I-trieu_chung_benh,0.7624,0.5756,0.6559,1522


{'precision': np.float64(0.7199786039047874),
 'recall': np.float64(0.7226845637583893),
 'f1': np.float64(0.7213290460878885),
 'accuracy': 0.9015952711604885,
 'f1_NNB': np.float64(0.3254817987152034)}

In [ ]:
import os
import torch

# Đường dẫn lưu model trên Drive
save_dir = "/content/drive/MyDrive/ALESRD_0.75"
os.makedirs(save_dir, exist_ok=True)

print("💾 Đang lưu Tokenizer và Trọng số mô hình kết hợp...")
tokenizer.save_pretrained(save_dir)

# Lưu toàn bộ state_dict của mô hình (bao gồm cả PhoBERT, Classifier, CRF và Focal Loss)
torch.save(trainer_entity_aware.model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))
print(f"✅ Đã lưu mô hình thành công tại: {save_dir}")

💾 Đang lưu Tokenizer và Trọng số mô hình kết hợp...
✅ Đã lưu mô hình thành công tại: /content/drive/MyDrive/ALESRD_0.75
